# 03 - Model training and evaluation

Baselines first, then the LSTM, all through one harness.

> **These notebooks define no functions.** Everything they call lives in `src/`.
> That rule is from `brain.md` section 7: logic written in a cell cannot be tested
> and silently drifts from the module, which is how report figures stop matching
> the code that ships.
>
> Until real case data is in `data/raw/cases/`, these fall back to a generated
> panel and every number describes a generator rather than dengue.

In [ ]:
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

from src.config import load_config

cfg = load_config("../config.yaml")
cfg.project.name, cfg.project.granularity

In [ ]:
from src.panel import assemble_panel
from src.preprocess import preprocess

try:
    panel = assemble_panel(cfg)
    SYNTHETIC = False
except Exception as error:
    print("no real data yet, using the synthetic stand-in:")
    print(" ", str(error).splitlines()[0])
    from src.synthetic import synthetic_panel
    panel = synthetic_panel(cfg)
    SYNTHETIC = True

clean = preprocess(panel, cfg).panel[list(panel.columns)]
clean.shape

In [ ]:
from src.features import build_features

data = build_features(clean, cfg)
X, y, spec = data
X.shape

## The folds

Rolling origin, cut on dates. Every state appears on both sides of every cut, and an embargo sits between train and test so no training label comes from the test window.

In [ ]:
from src.splits import rolling_origin

for fold in rolling_origin(spec.sample_index, cfg, horizon=spec.horizon):
    print(fold.describe())

## Baselines

The bar. If the LSTM cannot beat seasonal-naive, that is the finding.

In [ ]:
from src.evaluate import compare, run_experiment
from src.models.naive import baseline_factories

results = [
    run_experiment(factory, data, cfg, f'nb_{name}')
    for name, factory in baseline_factories(spec, cfg).items()
]
compare(results)[['mae_cases_per_100k', 'rmse_cases_per_100k', 'r2_log']]

## The LSTM

Same harness, no special-casing. Wrapped for conformal intervals, so coverage and CRPS come out too.

In [ ]:
from src.models.lstm import pooled_lstm
from src.uncertainty import conformal

lstm = run_experiment(conformal(pooled_lstm(spec, cfg), cfg), data, cfg, 'nb_lstm')
print(lstm.summary())

## The gate

Does the LSTM beat seasonal-naive? If not, stop and diagnose before building anything on top of it.

In [ ]:
seasonal = next(r for r in results if r.name.endswith('seasonal_naive'))
print(f'LSTM            {lstm.primary:.4f}')
print(f'seasonal naive  {seasonal.primary:.4f}')
print('PASS' if lstm.primary < seasonal.primary else 'FAIL - stop here')

## Interval calibration

An 80% interval should contain about 80% of held-out actuals.

In [ ]:
print(f"nominal  {lstm.mean['nominal_coverage']:.1%}")
print(f"observed {lstm.mean['coverage']:.1%} +/- {lstm.std['coverage']:.1%}")